# Лабораторная работа №1: Введение и воспроизводимость исследований

**Вариант №1** · датасет **Titanic** (`seaborn`) · задача **бинарная классификация** (таргет `survived`)

Цель лабораторной — воспроизводимый цикл исследования:
имитация «грязных» данных -> кастомный трансформер -> конвейер предобработки и модели -> экспорт.

## 1. Импорт библиотек

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
from pathlib import Path

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

import joblib

# Фиксируем воспроизводимость
SEED = 42
np.random.seed(SEED)

pd.set_option('display.max_columns', None)

# Корень проекта ищем по наличию .git (ноутбук лежит в src/notebooks)
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / '.git').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_RAW_DIR = PROJECT_ROOT / 'src' / 'data' / 'raw'
MODELS_DIR = PROJECT_ROOT / 'models'

## 2. Загрузка данных и имитация «грязных» значений

Загружаем датасет `titanic`, затем **искусственно вносим пропуски** (`np.nan`)
в **10% случайных значений** числовых колонок `age` и `fare`, чтобы сымитировать
реальные грязные данные. Результат сохраняем в `src/data/raw/titanic.csv`.

In [2]:
df = sns.load_dataset('titanic')

# Искусственно вносим пропуски в 10% случайных значений age и fare
dirty = df.copy()
for col in ['age', 'fare']:
    n_missing = int(len(dirty) * 0.10)
    idx = np.random.choice(dirty.index, size=n_missing, replace=False)
    dirty.loc[idx, col] = np.nan

print(f'Форма: {dirty.shape}')
print(f'Пропуски после внесения:\n{dirty[["age", "fare"]].isna().sum()}')

DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
dirty.to_csv(DATA_RAW_DIR / 'titanic.csv', index=False)
print(f'\nДатасет сохранён в {DATA_RAW_DIR / "titanic.csv"}')

Форма: (891, 15)
Пропуски после внесения:
age     252
fare     89
dtype: int64

Датасет сохранён в C:\OmSTU\Prodvinutie_metodi_7_semester\Pract_exercise\src\data\raw\titanic.csv


## 3. Кастомный трансформер «наличие пропусков в строке»

Трансформер наследуется от `BaseEstimator` и `TransformerMixin`.
Он создаёт бинарный признак: **1** — если в строке есть хотя бы один `NaN`,
**0** — если строка чистая. Признак добавляется колонкой `has_missing`,
а исходные фичи сохраняются для дальнейшей обработки.

In [3]:
class MissingRowIndicator(BaseEstimator, TransformerMixin):
    """Создаёт бинарный признак наличия пропусков в строке.

    Если в любой фиче строки есть хотя бы один NaN -> 1, иначе 0.
    Не уничтожает исходные признаки: добавляет колонку `has_missing`.
    """

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X['has_missing'] = X.isna().any(axis=1).astype(int)
        return X

## 4. Разделение на признаки, таргет и train/test split

Из фич исключаем колонку `alive` — она является прямой текстовой копией таргета
`survived` (это привело бы к «утечке» данных). Сплит: 80/20, `random_state=42`.

In [4]:
target = 'survived'
DROP = ['alive']  # утечка: текстовая копия таргета

features = [col for col in dirty.columns if col != target and col not in DROP]
X = dirty[features]
y = dirty[target]

print('Признаки:', features)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Признаки: ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'class', 'who', 'adult_male', 'deck', 'embark_town', 'alone']
Train: (712, 13), Test: (179, 13)


## 5. ColumnTransformer: предобработка числовых и категориальных фич

- **Числовые** (`age`, `sibsp`, `parch`, `fare`): `SimpleImputer(strategy='median')` + `StandardScaler`.
- **Категориальные** (`pclass`, `sex`, `embarked`, `class`, `who`, `adult_male`, `deck`, `embark_town`, `alone`): `SimpleImputer(strategy='most_frequent')` + `OneHotEncoder(handle_unknown='ignore')`.
- Признак `has_missing`, добавленный трансформером на шаге 3, пропускаем без изменений.

In [5]:
numeric_features = ['age', 'sibsp', 'parch', 'fare']
categorical_features = [
    'pclass', 'sex', 'embarked', 'class', 'who',
    'adult_male', 'deck', 'embark_town', 'alone',
]

numeric_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categ_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore')),
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_pipe, numeric_features),
        ('cat', categ_pipe, categorical_features),
        ('missing', 'passthrough', ['has_missing']),
    ]
)

print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['age', 'sibsp', 'parch', 'fare']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('ohe',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['pclass', 'sex', 'embarked', 'class', 'who',
                                  'adult_male', 'deck', 'embark_town',
                                  'alone']),
                                ('missing', 'passthrough', ['has_missing'])])


## 6. Сквозной Pipeline

Собираем общий конвейер:
`[Кастомный трансформер] -> [ColumnTransformer] -> [RandomForestClassifier]`.

In [6]:
model = Pipeline([
    ('missing_indicator', MissingRowIndicator()),
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=200, max_depth=6, random_state=SEED, n_jobs=-1
    )),
])

print(model)

Pipeline(steps=[('missing_indicator', MissingRowIndicator()),
                ('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'sibsp', 'parch',
                                                   'fare']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('ohe',
                                                

## 7. Обучение, оценка и экспорт

Обучаем пайплайн на тренировочной выборке, оцениваем **Accuracy** и **ROC-AUC**
на тестовой, сохраняем обученный пайплайн в `models/titanic_model.joblib`.

In [7]:
# Обучение
model.fit(X_train, y_train)

# Прогнозы
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# Метрики
acc = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)
print(f'Accuracy: {acc:.4f}')
print(f'ROC-AUC:  {roc_auc:.4f}')

# Экспорт обученного пайплайна
MODELS_DIR.mkdir(parents=True, exist_ok=True)
joblib.dump(model, MODELS_DIR / 'titanic_model.joblib')
print(f'\nПайплайн сохранён в {MODELS_DIR / "titanic_model.joblib"}')

Accuracy: 0.8212
ROC-AUC:  0.8555

Пайплайн сохранён в C:\OmSTU\Prodvinutie_metodi_7_semester\Pract_exercise\models\titanic_model.joblib


### Итог
Ноутбук прошёл полный цикл: загрузка и «загрязнение» данных, кастомный
трансформер, конвейер предобработки, обучение, оценка метрик и экспорт модели.
Датасет версионируется DVC (`dvc status`), код закоммичен в Git.